# Gradient Flow as an ODE and the Heavy-Ball Dynamics

Gradient descent $x^{k+1} = x^k - \tau \nabla f(x^k)$ is a forward-Euler discretisation of the **gradient flow ODE**:
$$
\dot{x}(t) = -\nabla f(x(t)).
$$
As the step $\tau \to 0$, the discrete trajectory converges to the continuous flow. The ODE viewpoint reveals stability conditions and helps design better algorithms.

## Gradient flow properties

For a $\mu$-strongly convex, $L$-smooth $f$:
- The flow contracts: $\|x(t) - x^*\| \le e^{-\mu t} \|x(0) - x^*\|$.
- The Lyapunov function $f(x(t)) - f^*$ decays as $e^{-2\mu t}$.
- Forward Euler (GD) with step $\tau < 2/L$ is **linearly stable**.

## Heavy-ball ODE

Adding a **friction term** $\alpha \dot{x}$ turns the first-order flow into the **second-order system**:
$$
\ddot{x}(t) + \alpha\, \dot{x}(t) + \nabla f(x(t)) = 0.
$$
This models a particle in a potential $f$ with viscous drag. Critical damping occurs at $\alpha = 2\sqrt{\mu}$. Underdamping ($\alpha < 2\sqrt{\mu}$) causes oscillations; overdamping ($\alpha > 2\sqrt{\mu}$) is sluggish.

## Nesterov ODE (Su–Boyd–Candès 2016)

The ODE limit of Nesterov's accelerated method is:
$$
\ddot{x}(t) + \frac{3}{t}\dot{x}(t) + \nabla f(x(t)) = 0.
$$
The time-varying friction $3/t$ decays to zero, so the system oscillates more and more freely, achieving $f(x(t)) - f^* = O(1/t^2)$.

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams['figure.dpi'] = 120

## 1D quadratic: ODE trajectories

For $f(x) = \frac{1}{2}\mu x^2$ the heavy-ball ODE $\ddot{x} + \alpha \dot{x} + \mu x = 0$ is a damped harmonic oscillator with exact solution $x(t) = e^{-\alpha t/2}(A\cos\omega t + B\sin\omega t)$ where $\omega = \sqrt{\mu - \alpha^2/4}$ (underdamped case).

In [ ]:
mu = 1.0
alpha_vals = [0.5, 2*np.sqrt(mu), 4.0]  # under-, critical, over-damped
cols = ['royalblue', 'seagreen', 'tomato']
labels = ['underdamped', 'critically damped', 'overdamped']

t_span = (0, 20); t_eval = np.linspace(0, 20, 500)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for alpha, col, lbl in zip(alpha_vals, cols, labels):
    def hb_ode(t, y, alpha=alpha):
        x, v = y
        return [v, -alpha*v - mu*x]
    sol = solve_ivp(hb_ode, t_span, [1.0, 0.0], t_eval=t_eval, dense_output=True)
    x_t = sol.y[0]
    axes[0].plot(t_eval, x_t, color=col, lw=2, label=f'$\\alpha={alpha:.2f}$ ({lbl})')
    axes[1].semilogy(t_eval, 0.5*mu*x_t**2, color=col, lw=2)

axes[0].axhline(0, color='k', lw=0.8, ls='--')
axes[0].set_xlabel('time $t$'); axes[0].set_ylabel('$x(t)$')
axes[0].set_title('Heavy-ball trajectories: $\\ddot{x} + \\alpha\\dot{x} + \\mu x = 0$')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].set_xlabel('time $t$'); axes[1].set_ylabel('$f(x(t))$ (log scale)')
axes[1].set_title('Energy decay'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2D: Phase portrait and gradient flow

For a 2D anisotropic quadratic $f(x,y) = \frac{1}{2}(\mu x^2 + Ly^2)$, we compare gradient flow with heavy-ball ODE.

In [ ]:
mu2, L2 = 1.0, 20.0
t_span2 = (0, 5); t_eval2 = np.linspace(0, 5, 800)

def gf_ode(t, y):
    return [-mu2*y[0], -L2*y[1]]

def hb2d_ode(t, y, alpha=2*np.sqrt(mu2)):
    x1, x2, v1, v2 = y
    return [v1, v2, -alpha*v1 - mu2*x1, -alpha*v2 - L2*x2]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

xi = np.linspace(-1.2, 1.2, 150); yi = np.linspace(-1.2, 1.2, 150)
X, Y = np.meshgrid(xi, yi)
F2 = 0.5*(mu2*X**2 + L2*Y**2)

for ax, alpha_hb in zip(axes, [None, 2*np.sqrt(mu2)]):
    ax.contourf(xi, yi, F2, levels=20, cmap='viridis', alpha=0.6)
    for x0y0 in [([1.0, 0.5, 0, 0], 'white'), ([0.8, -0.8, 0, 0], 'yellow')]:
        y0, col = x0y0
        if alpha_hb is None:
            sol = solve_ivp(gf_ode, t_span2, y0[:2], t_eval=t_eval2)
            ax.plot(sol.y[0], sol.y[1], '-', lw=2, color=col)
        else:
            sol = solve_ivp(lambda t,y: hb2d_ode(t, y, alpha_hb),
                            t_span2, y0, t_eval=t_eval2)
            ax.plot(sol.y[0], sol.y[1], '-', lw=2, color=col)
        ax.plot(y0[0], y0[1], 'o', ms=8, color=col)
    ax.plot(0, 0, 'r*', ms=12)
    ax.set_aspect('equal'); ax.axis('off')
    title = 'Gradient flow ODE' if alpha_hb is None else f'Heavy-ball ODE ($\\alpha={alpha_hb:.2f}$)'
    ax.set_title(title)

fig.suptitle(f'Continuous dynamics on $f = \\frac{{1}}{{2}}({mu2}x^2+{L2}y^2)$', y=1.02)
plt.tight_layout(); plt.show()

## Nesterov ODE vs gradient flow

We compare $f(x(t))$ for gradient flow, heavy-ball at critical damping, and the Nesterov ODE.

In [ ]:
T = 30; t_ev = np.linspace(0.01, T, 2000)

# Gradient flow
sol_gf = solve_ivp(gf_ode, (0.01, T), [1.0, 1.0], t_eval=t_ev)
fv_gf = 0.5*(mu2*sol_gf.y[0]**2 + L2*sol_gf.y[1]**2)

# Heavy-ball critical
alpha_c = 2*np.sqrt(mu2)
sol_hb = solve_ivp(lambda t,y: hb2d_ode(t,y,alpha_c), (0.01,T), [1,1,0,0], t_eval=t_ev)
fv_hb = 0.5*(mu2*sol_hb.y[0]**2 + L2*sol_hb.y[1]**2)

# Nesterov ODE: x'' + (3/t) x' + grad f = 0
def nes_ode(t, y):
    x1, x2, v1, v2 = y
    return [v1, v2, -(3/t)*v1 - mu2*x1, -(3/t)*v2 - L2*x2]
sol_nes = solve_ivp(nes_ode, (0.01, T), [1, 1, 0, 0], t_eval=t_ev)
fv_nes = 0.5*(mu2*sol_nes.y[0]**2 + L2*sol_nes.y[1]**2)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.semilogy(t_ev, fv_gf,  'royalblue', lw=2, label='Gradient flow $e^{-2\\mu t}$')
ax.semilogy(t_ev, fv_hb,  'tomato',    lw=2, label=f'Heavy-ball (critical $\\alpha={alpha_c:.2f}$)')
ax.semilogy(t_ev, fv_nes, 'seagreen',  lw=2, label='Nesterov ODE $O(1/t^2)$')
ax.semilogy(t_ev, fv_gf[0]*np.exp(-2*mu2*t_ev), 'royalblue', ls='--', lw=1, alpha=0.5, label='$e^{-2t}$ rate')
ax.semilogy(t_ev, 2.0/t_ev**2, 'seagreen', ls='--', lw=1, alpha=0.5, label='$2/t^2$ bound')
ax.set_xlabel('time $t$'); ax.set_ylabel('$f(x(t))$ (log scale)')
ax.set_title('Continuous-time dynamics: gradient flow vs. inertial methods')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Interactive: damping parameter

In [ ]:
def show_hb(alpha=2.0, T=25):
    t_ev = np.linspace(0.01, T, 1000)
    sol = solve_ivp(lambda t,y: hb2d_ode(t,y,alpha), (0.01,T), [1,1,0,0], t_eval=t_ev)
    fv = 0.5*(mu2*sol.y[0]**2 + L2*sol.y[1]**2)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    xi2 = np.linspace(-1.2, 1.2, 120); yi2 = np.linspace(-1.2, 1.2, 120)
    X2, Y2 = np.meshgrid(xi2, yi2)
    axes[0].contourf(xi2, yi2, 0.5*(mu2*X2**2+L2*Y2**2), levels=20, cmap='viridis', alpha=0.6)
    axes[0].plot(sol.y[0], sol.y[1], 'w-', lw=1.5)
    axes[0].plot(1, 1, 'ko', ms=8); axes[0].plot(0, 0, 'r*', ms=12)
    axes[0].set_aspect('equal'); axes[0].axis('off')
    axes[0].set_title(fr'Trajectory ($\alpha={alpha:.2f}$, $\alpha_c={2*np.sqrt(mu2):.2f}$)')
    axes[1].semilogy(t_ev, fv, 'royalblue', lw=2)
    axes[1].set_xlabel('time'); axes[1].set_ylabel('$f(x(t))$')
    axes[1].set_title('Energy decay'); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

interact(show_hb,
         alpha=FloatSlider(value=2.0, min=0.1, max=8.0, step=0.1, description='$\\alpha$'),
         T=IntSlider(value=25, min=5, max=60, step=5, description='$T$'));

## Bibliographical resources

- Polyak, B. T. (1964). Some methods of speeding up the convergence of iteration methods. *USSR Computational Mathematics and Mathematical Physics*, 4(5), 1–17.
- Su, W., Boyd, S. and Candès, E. J. (2016). A differential equation for modeling Nesterov's accelerated gradient method. *Journal of Machine Learning Research*, 17(153), 1–43.
- Wibisono, A., Wilson, A. C. and Jordan, M. I. (2016). A variational perspective on accelerated methods in optimization. *Proceedings of the National Academy of Sciences*, 113(47), E7351–E7358.
- Betancourt, M., Jordan, M. I. and Wilson, A. C. (2018). On symplectic optimization. arXiv:1802.03653.
- Muehlebach, M. and Jordan, M. I. (2019). A dynamical systems perspective on Nesterov acceleration. *ICML*, 6112–6121.